In [0]:
# MAGIC %pip install /Volumes/amfs_tm/raw/shared_libs/amfs_tp-0.1.0-py3-none-any.whl --force-reinstall

# COMMAND ----------
import sys
import os

# 1. Get the current directory of the notebook
# In Databricks, /Workspace/Repos/<user>/<repo_name>/notebooks
current_dir = os.getcwd()

# 2. Go up one level to the root and into 'src'
# We want to add /Workspace/Repos/<user>/<repo_name>/src
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))

if src_path not in sys.path:
    sys.path.append(src_path)
    print(f"Added {src_path} to Python path.")

    
import logging
from amfs_tp.pipeline.handler_job import HandlerJob

# Setup logging
logging.basicConfig(level=logging.INFO)

# COMMAND ----------

# 1. SETTINGS
snapshot = "202505"
catalog = "amfs_tm"

# COMMAND ----------

# 2. RUN HANDLER JOB
# This will clean and materialize tables into amfs_tm.clean
job = HandlerJob(spark, catalog=catalog)
job.run_cleaning(snapshot)

# COMMAND ----------

# 3. VERIFICATION (Manual Check)
print(f"--- Data Quality Verification for {snapshot} ---")

# Check a critical BM table
print("\nSample: PHSumm (Clean)")
display(spark.table(f"{catalog}.clean.bm_phsumm").limit(5))

# Check a critical AMFS table
print("\nSample: Call Tracking (Clean)")
display(spark.table(f"{catalog}.clean.amfs_call_tracking").limit(5))

# COMMAND ----------

# 4. DOWNSTREAM STEPS (COMMENTED OUT FOR TESTING)
# The steps below are for the next phase of the pipeline.

# print("Step: Feature Engineering...")
# feature_job = FeatureJob(spark, snapshot=snapshot, catalog=catalog)
# feature_job.run()

# print("Step: Model Scoring...")
# scoring_job = ScoringJob(spark, snapshot=snapshot, catalog=catalog)
# scoring_job.run()